# U.S. Retail Sales: Time Series Analysis in SQL
DSA 508 | Jacob Willson

Dataset: Census Bureau Monthly Retail Trade Survey, `us_retail_sales.csv` (1992-01 to 2020-12, millions of USD, not seasonally adjusted).

All analysis runs in PostgreSQL (Azure Database for PostgreSQL). SQL lives in `sql/`; this notebook executes it, charts the results and interprets them.

## Key results at a glance

| # | Result | Where |
|---|---|---|
| 1 | E-commerce grew 11.7%/yr (1992-2019) and rose from 1.9% to 15.2% of U.S. retail; it passed department stores in 2007 and was 7.5x their size by 2020 | Steps 2, 5.1, 5.2 |
| 2 | Department stores peaked in 2000 and fell in 19 of the next 20 years; confirmed trend break Nov-Dec 2001 | Steps 2, 3 |
| 3 | Book stores peaked in 2007; the post-2008 decline never reversed; 2020 sales were 62.6% below peak | Steps 2, 3, 5.4 |
| 4 | Warehouse clubs never declined but slowed in steps: 13.4%/yr, then 9.4% (2004), then 3.3% (mid-2009) | Step 3 (PELT) |
| 5 | Gasoline stations follow oil prices: 9 confirmed turns, the most volatile growth of the five | Steps 3, 5.9 |
| 6 | Department stores forecast for 2023: $113B (linear trend, 95% interval $104B to $121B); $96B if 2020 does not recover | Step 4 |
| 7 | Recessions ratchet decliners down: department and book stores never regained their pre-2008 level | Step 5.4 |
| 8 | The December spike is shrinking in every holiday category (STL: department stores +86% to +67% vs trend) | Steps 5.3, 5.11 |
| 9 | Calendar noise is large: one extra weekend day vs last year cuts e-commerce YoY by 2.7 points; Thanksgiving timing moves Cyber Monday between November and December | Step 5.10 |

## Step 1: Connect and load `retail_sales`

In [ ]:
import os, re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

load_dotenv()  # reads .env in the project folder

url = URL.create(
    "postgresql+psycopg",
    username=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"],   # URL.create handles special characters
    host=os.environ["PGHOST"],
    port=int(os.getenv("PGPORT", 5432)),
    database=os.environ["PGDATABASE"],
    query={"sslmode": os.getenv("PGSSLMODE", "require")},
)
engine = create_engine(url)

SQL_DIR = Path("sql")
CSV_PATH = Path("data/us_retail_sales.csv")
pd.set_option("display.max_colwidth", 60)

In [ ]:
def load_queries(filename):
    """Split a .sql file into {name: sql} using '-- name: <id>' markers."""
    raw = (SQL_DIR / filename).read_text()
    parts = re.split(r"^-- name:\s*(\S+)\s*$", raw, flags=re.M)
    return {parts[i]: parts[i + 1].strip() for i in range(1, len(parts), 2)}

def run(sql):
    """Run a SELECT and return a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def execute(sql):
    """Run DDL/DML (may contain several statements)."""
    # raw psycopg cursor with no parameters, so a '%' in SQL comments is not
    # mistaken for a placeholder
    with engine.begin() as conn:
        conn.connection.cursor().execute(sql)

q = load_queries("01_load.sql")
list(q)

Connection check. `ssl = True` confirms the Azure connection is encrypted.

In [ ]:
run("""
SELECT current_database() AS db,
       split_part(version(), ' on ', 1) AS server,
       (SELECT ssl FROM pg_stat_ssl WHERE pid = pg_backend_pid()) AS ssl
""")

Create the table if needed, then bulk load the CSV with `COPY ... FROM STDIN` only when the table is empty (streams the local file to the server, so it also works against Azure).

In [ ]:
execute(q["create_table"])   # CREATE TABLE IF NOT EXISTS: safe to rerun

n_rows = run("SELECT COUNT(*) AS n FROM retail_sales").n[0]
if n_rows == 0:
    with engine.begin() as conn:
        cur = conn.connection.cursor()   # raw psycopg cursor for COPY
        with CSV_PATH.open("rb") as f, cur.copy(
            "COPY retail_sales FROM STDIN WITH (FORMAT csv, HEADER true)"
        ) as copy:
            while chunk := f.read(1 << 16):
                copy.write(chunk)
    print("Loaded:", run("SELECT COUNT(*) AS n FROM retail_sales").n[0], "rows")
else:
    print(f"Table already loaded ({n_rows} rows); skipping COPY")

### Sanity checks
Expected: 22,620 rows, 65 categories, 348 months; 594 null sales (492 Not Available, 102 Supressed).

In [ ]:
run(q["check_row_count"])

In [ ]:
run(q["check_nulls"])

In [ ]:
run(q["check_null_categories"])

The five analysis categories must be complete (348 months, 0 nulls).

In [ ]:
run(q["check_candidates"])

## Step 2: Monthly and yearly trends (window functions)

All metrics come from `sql/02_trends.sql`. The view `v_monthly_metrics` keeps one row per category per month and attaches running totals, a 12-month moving average, YoY change and share of year with `PARTITION BY kind_of_business ORDER BY sales_month`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

q2 = load_queries("02_trends.sql")
execute(q2["create_view_monthly"])   # DROP VIEW IF EXISTS + CREATE VIEW: safe to rerun

# Fixed color per category (validated categorical palette, same order everywhere)
CATS = {
    "Electronic shopping and mail-order houses": ("E-commerce",        "#2a78d6"),
    "Warehouse clubs and superstores":           ("Warehouse clubs",   "#eb6834"),
    "Gasoline stations":                         ("Gasoline stations", "#1baf7a"),
    "Department stores":                         ("Department stores", "#eda100"),
    "Book stores":                               ("Book stores",       "#e87ba4"),
}
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10, "axes.edgecolor": GRID, "axes.labelcolor": INK2,
    "xtick.color": INK2, "ytick.color": INK2, "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlecolor": INK,
})
Path("figures").mkdir(exist_ok=True)

monthly = run("SELECT * FROM v_monthly_metrics ORDER BY kind_of_business, sales_month")
monthly["sales_month"] = pd.to_datetime(monthly["sales_month"])
for c in ["sales", "ma_12", "ma_3", "yoy_pct", "pct_of_year", "running_total"]:
    monthly[c] = monthly[c].astype(float)
monthly.shape

### Book stores example: every window metric side by side

In [ ]:
run(q2["example_book_stores"])

### Chart 1: Monthly sales with 12-month moving average
Thin line = raw monthly sales (seasonal spikes). Bold line = 12-month moving average, which cancels seasonality and shows the trend. Each panel has its own y-scale because the categories differ in size by 100x.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 12), sharex=True)
for ax, (cat, (label, color)) in zip(axes, CATS.items()):
    d = monthly[monthly.kind_of_business == cat]
    ax.plot(d.sales_month, d.sales / 1000, color=color, lw=0.8, alpha=0.45, label="Monthly sales")
    ax.plot(d.sales_month, d.ma_12 / 1000, color=color, lw=2.2, label="12-month moving avg")
    ax.set_title(label, loc="left")
    ax.set_ylabel("$B / month")
    fmt = "{:,.1f}" if d.sales.max() < 10_000 else "{:,.0f}"
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _, f=fmt: f.format(v)))
axes[0].legend(loc="upper left", frameon=False)
fig.suptitle("Monthly sales and 12-month moving average, 1992-2020", x=0.01, ha="left",
             fontsize=13, fontweight="bold", color=INK)
fig.tight_layout()
fig.savefig("figures/01_monthly_ma12.png", dpi=150, bbox_inches="tight")
plt.show()

### Yearly totals, YoY growth and growth index

In [ ]:
yearly = run(q2["yearly_totals"])
for c in ["year_sales", "yoy_pct", "index_1992", "pct_from_peak"]:
    yearly[c] = yearly[c].astype(float)
yearly.head(8)

### Chart 2: Yearly sales indexed to 1992 = 100
Indexing puts a $8B category and an $850B category on one axis. Log scale so equal growth rates look like equal slopes. The dashed line marks 100 (the 1992 level).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for cat, (label, color) in CATS.items():
    d = yearly[yearly.kind_of_business == cat]
    ax.plot(d.sales_year, d.index_1992, color=color, lw=2, marker="o", ms=3.5, label=label)
    last = d.iloc[-1]
    ax.annotate(f"{label}  {last.index_1992:,.0f}", (last.sales_year, last.index_1992),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=9, color=INK)
ax.set_yscale("log")
ax.axhline(100, color=INK2, lw=1, ls="--")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.set_yticks([50, 100, 200, 500, 1000, 2000])
ax.set_xlim(1991.5, 2026)
ax.set_xticks(range(1992, 2021, 4))
ax.set_ylabel("Index (1992 = 100), log scale")
ax.set_title("Yearly sales indexed to 1992", loc="left", fontsize=13)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
fig.savefig("figures/02_yearly_index.png", dpi=150, bbox_inches="tight")
plt.show()

### Category summary: size, long-run growth, peak, COVID effect

In [ ]:
summary = run(q2["category_summary"]); summary

### Chart 3: Seasonality, average share of annual sales by month (1992-2019)
8.33% = a perfectly flat year (gray). Red = above-average month, blue = below.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

seas = run(q2["seasonality"]).set_index("month_num").astype(float)
seas.columns = ["E-commerce", "Department stores", "Book stores", "Warehouse clubs", "Gasoline stations"]
grid = seas.T
cmap = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])
norm = TwoSlopeNorm(vmin=5.5, vcenter=100 / 12, vmax=14.5)

fig, ax = plt.subplots(figsize=(10, 3.6))
im = ax.imshow(grid.values, cmap=cmap, norm=norm, aspect="auto")
months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
ax.set_xticks(range(12), months); ax.set_yticks(range(len(grid)), grid.index)
ax.grid(False)
for i in range(grid.shape[0]):
    for j in range(12):
        v = grid.values[i, j]
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8.5,
                color="white" if abs(v - 8.33) > 3.5 else INK)
for s in ax.spines.values(): s.set_visible(False)
ax.set_title("Share of annual sales by month (%), flat year = 8.3%", loc="left", fontsize=13)
fig.colorbar(im, ax=ax, shrink=0.8, label="% of year", ticks=[6, 8.33, 10, 12, 14]).ax.set_yticklabels(["6", "8.3", "10", "12", "14"])
fig.tight_layout()
fig.savefig("figures/03_seasonality.png", dpi=150, bbox_inches="tight")
plt.show()

### Step 2 findings

**Growth and decline**

| Category | 1992 | 2019 | CAGR 1992-2019 | Peak year | 2020 vs 2019 |
|---|---|---|---|---|---|
| E-commerce | $35.2B | $705.5B | +11.7% | 2020 | +20.4% |
| Warehouse clubs | $40.0B | $496.2B | +9.8% | 2020 | +7.5% |
| Gasoline stations | $156.3B | $501.1B | +4.4% | 2012 | -14.7% |
| Book stores | $8.3B | $8.8B | +0.2% | 2007 | -27.4% |
| Department stores | $177.1B | $135.0B | -1.0% | 2000 | -16.2% |
| *Total retail (benchmark)* | | | *+4.2%* | | |

- **E-commerce** grew every year from 1992 to 2020. Slowest years: 2001 (+0.8%) and 2008-2009 (+2.5%, +2.8%).
- **Warehouse clubs** grew 12-24% a year in the 1990s and 2-5% a year after 2009.
- **Gasoline stations** swing with oil prices: -22.3% (2009), -17.6% (2015), -14.7% (2020).
- **Book stores**: the near-zero CAGR hides a rise to a $17.2B peak in 2007, then decline: -10.0% (2011), -10.5% (2012). 2020 was 62.6% below peak.
- **Department stores** peaked at $232.5B in 2000 and declined in 19 of the 20 years after (only 2004 grew). Already 41.9% below peak in 2019.

**Seasonality** (average share of annual sales, 1992-2019; a flat year = 8.3% per month)

| Category | Peak month(s) | Lowest month | December share |
|---|---|---|---|
| Department stores | Dec 14.4%, Nov 10.0% | Jan 6.4% | 14.4% |
| Book stores | Aug 12.6%, Dec 12.6%, Jan 12.4% | Apr 6.2% | 12.6% |
| E-commerce | Dec 12.3%, Nov 10.0% | Feb 7.0% | 12.3% |
| Warehouse clubs | Dec 11.5%, Nov 9.2% | Jan/Feb 7.1% | 11.5% |
| Gasoline stations | Aug 9.1%, Jul 9.0% | Feb 7.1% | 8.1% |

- Department stores are the most holiday-dependent category; gasoline is the only one with no December peak.
- Book stores have three peaks. January and August line up with the start of college semesters (textbook hypothesis, see Business insights).
- Gasoline peaks in the summer driving season.

**Differences between categories**
- The five split into two growers (e-commerce, warehouse clubs), two decliners (department and book stores) and one price-driven series (gasoline).
- Scale differs by 100x (book stores under $10B vs e-commerce over $800B), which is why Chart 2 indexes to 1992 = 100 on a log scale.

## Step 3: Change-point detection

All detection runs in SQL (`sql/03_change_points.sql`) on de-seasonalized signals, so December spikes never count as change points:

| Method | Signal | Rule |
|---|---|---|
| **A. Moving-average crossover** | `ma_12` vs `ma_36` | fast trend line crosses the slow one |
| **B. Local slope reversal** | `REGR_SLOPE(ma_12)` over trailing 24 months | slope changes sign **and holds for 6 months** |
| **C. Standard-deviation shock** | monthly YoY % vs its trailing 36-month mean and SD | \|z\| ≥ 3, consecutive months merged into one episode |

A change point is **confirmed** when at least two methods fire in the same direction within 12 months. PELT (a formal penalized change-point algorithm) is run in Python as an independent cross-check.

In [ ]:
q3 = load_queries("03_change_points.sql")
execute(q3["create_view_signals"])
execute(q3["create_view_change_points"])

SHORT = {cat: label for cat, (label, _) in CATS.items()}
cp_all = run(q3["change_points_all"])
cp_all["kind_of_business"] = cp_all["kind_of_business"].map(SHORT)
run(q3["change_points_count"]).replace(SHORT)

### Confirmed change-point events (2+ methods agree)

In [ ]:
confirmed = run(q3["change_points_confirmed"])
confirmed["kind_of_business"] = confirmed["kind_of_business"].map(SHORT)
confirmed

### YoY shocks (method C), including single-method flags

In [ ]:
cp_all[cp_all.method.str.startswith("C")].reset_index(drop=True)

### Trend growth by era (local slope, % per year)
Captures slowdowns that never flip sign, which methods A and B cannot see.

In [ ]:
run(q3["trend_by_era"]).replace(SHORT)

In [ ]:
signals = run("SELECT * FROM v_change_signals ORDER BY kind_of_business, sales_month")
signals["sales_month"] = pd.to_datetime(signals["sales_month"])
for c in ["sales", "ma_12", "ma_36", "trend_pct_yr", "yoy_pct", "yoy_z"]:
    signals[c] = signals[c].astype(float)
confirmed["event_start"] = pd.to_datetime(confirmed["event_start"])
confirmed["last_signal"] = pd.to_datetime(confirmed["last_signal"])
cp_all["sales_month"] = pd.to_datetime(cp_all["sales_month"])

### Chart 4: Change points on the 12-month trend
Bold = 12-month moving average, dashed gray = 36-month moving average. Shaded bands = confirmed events (▼ down, ▲ up). × = YoY shock (method C) on its own.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 13), sharex=True)
for ax, (cat, (label, color)) in zip(axes, CATS.items()):
    d = signals[signals.kind_of_business == cat]
    ax.plot(d.sales_month, d.ma_12 / 1000, color=color, lw=2.2, label="12-mo MA")
    ax.plot(d.sales_month, d.ma_36 / 1000, color=INK2, lw=1.2, ls="--", label="36-mo MA")
    ev = confirmed[confirmed.kind_of_business == label]
    for _, e in ev.iterrows():
        ax.axvspan(e.event_start, e.last_signal + pd.DateOffset(months=1), color=INK2, alpha=0.15, lw=0)
        y = d.loc[d.sales_month == e.event_start, "ma_12"].iloc[0] / 1000
        ax.plot(e.event_start, y, marker="v" if e.direction == "down" else "^",
                color=INK, ms=9, mec="white", mew=1.2, zorder=5)
    sh = cp_all[(cp_all.kind_of_business == label) & cp_all.method.str.startswith("C")]
    sh = sh.merge(d[["sales_month", "ma_12"]], on="sales_month")
    ax.plot(sh.sales_month, sh.ma_12 / 1000, "x", color=INK, ms=8, mew=1.8, zorder=5)
    ax.set_title(f"{label}  ({len(ev)} confirmed events, {len(sh)} shocks)", loc="left")
    ax.set_ylabel("$B / month")
from matplotlib.lines import Line2D
axes[0].legend(handles=[
    Line2D([], [], color=INK, lw=2.2, label="12-mo MA (category color)"),
    Line2D([], [], color=INK2, lw=1.2, ls="--", label="36-mo MA"),
    Line2D([], [], marker="v", color=INK, lw=0, ms=8, label="Confirmed turn down"),
    Line2D([], [], marker="^", color=INK, lw=0, ms=8, label="Confirmed turn up"),
    Line2D([], [], marker="x", color=INK, lw=0, ms=8, mew=1.8, label="YoY shock |z| >= 3"),
], frameon=False, loc="upper left", fontsize=8.5)
fig.suptitle("Detected change points, 1992-2020", x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK)
fig.tight_layout()
fig.savefig("figures/04_change_points.png", dpi=150, bbox_inches="tight")
plt.show()

### Cross-check: PELT on the local slope
PELT (Pruned Exact Linear Time, `ruptures` library) finds the split points that best divide each category's trend-growth series into segments with different means, penalized so it does not over-split (penalty = 2·ln(n)·variance, minimum segment 24 months). It is a formal statistical method, independent of the SQL rules above.

In [ ]:
import numpy as np
import ruptures as rpt

pelt_rows = []
for cat, (label, _) in CATS.items():
    d = signals[(signals.kind_of_business == cat) & signals.trend_pct_yr.notna()].reset_index(drop=True)
    x = d.trend_pct_yr.values
    bkps = rpt.Pelt(model="l2", min_size=24).fit(x).predict(pen=2 * np.log(len(x)) * np.var(x))
    start = 0
    for b in bkps:
        pelt_rows.append({"category": label,
                          "from": d.sales_month[start].strftime("%Y-%m"),
                          "to": d.sales_month[b - 1].strftime("%Y-%m"),
                          "mean_trend_pct_yr": round(x[start:b].mean(), 1)})
        start = b
pelt = pd.DataFrame(pelt_rows)
pelt

### Chart 5: Local slope (trend growth, % per year) with PELT regimes
Thin line = SQL rolling 24-month slope of the 12-month MA. Dark steps = PELT segment means. Below the dashed zero line the category is shrinking.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 12), sharex=True)
for ax, (cat, (label, color)) in zip(axes, CATS.items()):
    d = signals[(signals.kind_of_business == cat) & signals.trend_pct_yr.notna()]
    ax.plot(d.sales_month, d.trend_pct_yr, color=color, lw=1.8)
    for _, r in pelt[pelt.category == label].iterrows():
        x0, x1 = pd.to_datetime(r["from"]), pd.to_datetime(r["to"]) + pd.DateOffset(months=1)
        ax.hlines(r.mean_trend_pct_yr, x0, x1, color=INK, lw=2.4)
        ax.annotate(f"{r.mean_trend_pct_yr:+.1f}%", (x0 + (x1 - x0) / 2, r.mean_trend_pct_yr),
                    xytext=(0, 6), textcoords="offset points", ha="center", fontsize=8.5, color=INK)
    ax.axhline(0, color=INK2, lw=1, ls="--")
    ax.set_title(label, loc="left")
    ax.set_ylabel("% per year")
axes[0].legend(handles=[
    Line2D([], [], color=INK2, lw=1.8, label="Local slope, rolling 24 mo (category color)"),
    Line2D([], [], color=INK, lw=2.4, label="PELT regime mean"),
], frameon=False, loc="lower left", fontsize=8.5)
fig.suptitle("Trend growth regimes, 1994-2020", x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK)
fig.tight_layout()
fig.savefig("figures/05_trend_regimes.png", dpi=150, bbox_inches="tight")
plt.show()

### Step 3 findings

**Detection summary**

| Category | A. Crossover | B. Slope reversal | C. YoY shock | Confirmed events |
|---|---|---|---|---|
| Gasoline stations | 9 | 9 | 5 | 9 |
| Department stores | 3 | 3 | 4 | 3 |
| Book stores | 3 | 3 | 4 | 3 |
| E-commerce | 0 | 0 | 8 | 0 |
| Warehouse clubs | 0 | 0 | 6 | 0 |

**Confirmed change points** (2+ methods agree within 12 months)

| Category | Start | Direction | Methods | Likely driver *(hypothesis)* |
|---|---|---|---|---|
| Department stores | 2001-11 | down | A + B | 2001 recession after the 2000 peak |
| Department stores | 2005-04 | up | A + B | short recovery |
| Department stores | 2006-01 | down | A + B | decline resumes; no upturn since |
| Book stores | 2007-01 | down | A + B | peak reached |
| Book stores | 2007-10 | up | A + B | short bounce |
| Book stores | 2008-11 | down | A + B | Great Recession; decline never reversed |
| Gasoline stations | 1998-09 | down | A + B | 1998 oil price collapse |
| Gasoline stations | 1999-07 | up | A + B + C | oil price rebound |
| Gasoline stations | 2002-05 | down | A + B | post-2001 price slump |
| Gasoline stations | 2002-12 | up | A + B | 2003-2008 price run-up |
| Gasoline stations | 2008-11 | down | A + B + C | 2008 oil crash |
| Gasoline stations | 2010-12 | up | A + B | price recovery |
| Gasoline stations | 2014-02 | down | A + B + C | 2014-15 oil glut |
| Gasoline stations | 2017-08 | up | A + B | price recovery |
| Gasoline stations | 2020-03 | down | A + B + C | COVID demand collapse |

**Growth categories: slowdowns, not reversals** (PELT regimes, local slope in % per year)

| Category | Regimes |
|---|---|
| Warehouse clubs | +13.4% to 2003, +9.4% 2004 to mid-2009, +3.3% after |
| E-commerce | +12.6%, dips to +4.6% (2002-03) and +3.6% (2008-10), back to +8.6% and +10.8% |
| Department stores | +3.1% to mid-2001, negative in every regime after (-1.4% to -5.7%) |
| Book stores | +6.7% to 2001, +2.0% to mid-2009, then -7.9%, -2.9%, -11.7% |

- A and B cannot fire for a series that never turns down, so e-commerce and warehouse clubs show only shocks (C) and PELT slowdowns.

**Method comparison**
- A (crossover) usually fires first; B (slope) confirms 1-5 months later (Department stores Nov then Dec 2001; Gasoline Apr then Aug 2009). One exception: Department stores 2005, where B came first.
- C (shock) is earliest for sudden events: Gasoline shock Nov 2008 vs crossover Apr 2009.
- PELT lands close to the SQL turns: Department stores mid-2001 (SQL Nov 2001), Book stores mid-2009 (SQL Nov 2008 to Mar 2009).
- Single-month shocks are often calendar effects. E-commerce Nov 2019 (+4.9% YoY) and Dec 2019 (+26.7%) follow Thanksgiving on Nov 28, 2019, which moved Cyber Monday into December (see 5.10). E-commerce Dec 2018 (-0.9%) has no calendar explanation.
- Book stores' largest non-COVID shock: Dec 2011, YoY -25.5% (z = 3.8), the first holiday season after Borders liquidated.

## Step 4: Three-year forecast (2021-2023)

**Primary category: Department stores.** Every method also runs for the other four categories so the results can be compared across a steady decline, steady growth and a volatile, price-driven series.

All forecasts are computed in SQL (`sql/04_forecast.sql`); Holt-Winters is added in Python as an independent benchmark.

| Method | How it works | SQL |
|---|---|---|
| **M1 Linear trend** | straight line through annual sales, with a 95% prediction interval | `REGR_SLOPE`, `REGR_INTERCEPT`, `REGR_SXX`, `REGR_SYY` |
| **M2 Log-linear trend** | constant % change per year (straight line through ln(sales)) | `REGR_SLOPE(LN(sales), year)` |
| **M3 Rolling mean, flat** | last 12-month moving average held constant | `ma_12` at Dec 2019 |
| **M4 Rolling mean + drift** | 12-month MA extended by its average change over the last 2 years | `ma_12` Dec 2019 vs Dec 2017 |
| **M5 Rebased trend** | M1's slope restarted from the 2020 actual level | M1 slope + 2020 actual |
| **HW Holt-Winters** | damped additive trend, multiplicative seasonality, monthly | `statsmodels` (Python) |

**Fit window: 2010-2019.** It starts after the Great Recession and stops before COVID, so neither shock bends the trend. 2020 is therefore a genuine out-of-sample year, and M1 vs M5 brackets the two COVID scenarios: *return to trend* vs *no recovery*. The window choice itself is stress-tested below.

In [ ]:
q4 = load_queries("04_forecast.sql")
execute(q4["create_view_yearly"])

def num(df):
    """Convert Decimal/object columns from Postgres to floats where possible."""
    for c in df.columns:
        if df[c].dtype == object:
            try:
                df[c] = pd.to_numeric(df[c])
            except (ValueError, TypeError):
                pass
    return df

def short(df, col="kind_of_business"):
    df[col] = df[col].map(SHORT).fillna(df[col])
    return df

DEPT = "Department stores"
METHOD_COLORS = {"M1 Linear": "#2a78d6", "M2 Log-linear": "#eb6834", "M3 Flat MA": "#1baf7a",
                 "M4 MA + drift": "#eda100", "M5 Rebased": "#e87ba4", "Holt-Winters": "#4a3aa7"}

fit = short(num(run(q4["trend_fit"])))
fit

### Annual forecasts, all methods

In [ ]:
fa = short(num(run(q4["forecast_annual"])))
fa[fa.kind_of_business == DEPT]

In [ ]:
# Full table for all categories
fa

### COVID gap: 2020 actual vs the 2010-2019 trend

In [ ]:
short(num(run(q4["forecast_covid_gap"])))

### Holt-Winters benchmark (Python)
Fitted on monthly data 2010-2019, forecasting 48 months (2020-2023). A second fit on 2010-2016 gives a backtest comparable to the SQL methods.

In [ ]:
import warnings
from statsmodels.tsa.holtwinters import ExponentialSmoothing

hw_annual, hw_monthly, hw_bt = [], {}, []
for cat, (label, _) in CATS.items():
    s = (monthly[monthly.kind_of_business == cat]
         .set_index("sales_month")["sales"].asfreq("MS"))
    actual_yr = s.groupby(s.index.year).sum()
    for train_end, horizon_label in [("2019-12-01", "forecast"), ("2016-12-01", "backtest")]:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = ExponentialSmoothing(s["2010-01-01":train_end], trend="add", damped_trend=True,
                                         seasonal="mul", seasonal_periods=12).fit()
        fc = model.forecast(48)
        fc_yr = fc.groupby(fc.index.year).sum()
        if horizon_label == "forecast":
            hw_monthly[label] = fc
            for yr in range(2020, 2024):
                hw_annual.append({"kind_of_business": label, "forecast_year": yr,
                                  "hw": round(fc_yr[yr])})
        else:
            hw_bt.append({"kind_of_business": label,
                          "mape_hw": round(100 * np.mean([abs(fc_yr[y] / actual_yr[y] - 1)
                                                          for y in (2017, 2018, 2019)]), 2)})
hw_annual = pd.DataFrame(hw_annual)
hw_bt = pd.DataFrame(hw_bt)
fa = fa.merge(hw_annual, on=["kind_of_business", "forecast_year"], how="left")
fa[fa.kind_of_business == DEPT]

### Chart 6: Department stores, annual forecast 2021-2023
Black = actual. Thin blue = M1 fitted on 2010-2019. Shaded band = M1 95% prediction interval. Each method is labeled at 2023.

In [ ]:
d_hist = yearly[(yearly.kind_of_business == DEPT) & (yearly.sales_year >= 2000)]
d_fa = fa[fa.kind_of_business == DEPT].sort_values("forecast_year")
d_fit = fit[fit.kind_of_business == DEPT].iloc[0]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(d_hist.sales_year, d_hist.year_sales / 1000, color=INK, lw=2, marker="o", ms=4, label="Actual")
yrs_fit = np.arange(2010, 2020)
b = d_fit.slope_musd_per_yr
a = d_fa.m1_linear.iloc[0] - b * 2020
ax.plot(yrs_fit, (a + b * yrs_fit) / 1000, color=METHOD_COLORS["M1 Linear"], lw=1, alpha=0.8)
ax.fill_between(d_fa.forecast_year, d_fa.m1_lower_95 / 1000, d_fa.m1_upper_95 / 1000,
                color=METHOD_COLORS["M1 Linear"], alpha=0.12, lw=0, label="M1 95% interval")
series = [("M1 Linear", "m1_linear"), ("M2 Log-linear", "m2_log_linear"), ("M3 Flat MA", "m3_flat_ma"),
          ("M4 MA + drift", "m4_ma_drift"), ("M5 Rebased", "m5_rebased"), ("Holt-Winters", "hw")]
end_labels = []
for name, col in series:
    ax.plot(d_fa.forecast_year, d_fa[col] / 1000, color=METHOD_COLORS[name], lw=2,
            marker="o", ms=4, ls="--", label=name)
    end_labels.append((d_fa[col].iloc[-1] / 1000, name))
# de-overlap end labels
end_labels.sort()
placed = []
for y, name in end_labels:
    y_adj = max(y, placed[-1] + 4.8) if placed else y
    placed.append(y_adj)
    ax.annotate(f"{name}  {y:,.0f}", (2023, y), xytext=(2023.25, y_adj), fontsize=8.5, color=INK,
                va="center", arrowprops=dict(arrowstyle="-", color=GRID, lw=0.8))
a20 = d_hist[d_hist.sales_year == 2020].year_sales.iloc[0] / 1000
ax.annotate(f"2020 actual {a20:,.0f}\n(COVID)", (2020, a20), xytext=(2016.3, 100),
            fontsize=8.5, color=INK, arrowprops=dict(arrowstyle="->", color=INK2, lw=0.8))
ax.axvspan(2010, 2019.5, color=GRID, alpha=0.35, lw=0)
ax.text(2010.2, 238, "fit window 2010-2019", fontsize=8.5, color=INK2)
ax.set_xlim(1999.5, 2027.5)
ax.set_ylim(80, 245)
ax.set_xticks(range(2000, 2024, 2))
ax.set_ylabel("Annual sales ($B)")
ax.set_title("Department stores: actual and forecast by method", loc="left", fontsize=13)
ax.legend(frameon=False, loc="lower left", fontsize=8.5, ncol=2)
fig.tight_layout()
fig.savefig("figures/06_dept_forecast_annual.png", dpi=150, bbox_inches="tight")
plt.show()

### Monthly forecast with seasonality (M1 annual trend x 2015-2019 monthly shares)

In [ ]:
fm = short(num(run(q4["forecast_monthly"])))
fm["sales_month"] = pd.to_datetime(fm["sales_month"])
fm[fm.kind_of_business == DEPT].head(12)

### Chart 7: Department stores, monthly actual and forecast
Shaded band = 95% interval on the M1-based monthly forecast. Purple dashed = Holt-Winters (trained through 2019, so 2020 shows its COVID miss).

In [ ]:
d_act = monthly[(monthly.kind_of_business == [k for k, v in CATS.items() if v[0] == DEPT][0])
               & (monthly.sales_month >= "2015-01-01")]
d_fm = fm[fm.kind_of_business == DEPT]
hw_d = hw_monthly[DEPT]

fig, ax = plt.subplots(figsize=(11, 5.2))
ax.plot(d_act.sales_month, d_act.sales / 1000, color=INK, lw=1.4, label="Actual")
ax.fill_between(d_fm.sales_month, d_fm.lower_95 / 1000, d_fm.upper_95 / 1000,
                color=METHOD_COLORS["M1 Linear"], alpha=0.15, lw=0, label="M1 monthly 95% interval")
ax.plot(d_fm.sales_month, d_fm.forecast / 1000, color=METHOD_COLORS["M1 Linear"], lw=1.8,
        label="M1 trend x seasonal shares")
ax.plot(hw_d.index, hw_d.values / 1000, color=METHOD_COLORS["Holt-Winters"], lw=1.4, ls="--",
        label="Holt-Winters (trained to 2019)")
ax.axvline(pd.Timestamp("2021-01-01"), color=INK2, lw=0.8)
ax.text(pd.Timestamp("2021-02-01"), ax.get_ylim()[1] * 0.97, "forecast →", fontsize=9, color=INK2, va="top")
ax.set_ylabel("Monthly sales ($B)")
ax.set_title("Department stores: monthly sales and 2021-2023 forecast", loc="left", fontsize=13)
ax.legend(frameon=False, loc="upper right", fontsize=8.5)
fig.tight_layout()
fig.savefig("figures/07_dept_forecast_monthly.png", dpi=150, bbox_inches="tight")
plt.show()

### Backtest: fit 2010-2016, forecast 2017-2019, compare to actual (MAPE %)

In [ ]:
bt = short(num(run(q4["forecast_backtest"]))).merge(hw_bt, on="kind_of_business")
bt["best_method"] = bt.drop(columns="kind_of_business").idxmin(axis=1)
bt

In [ ]:
# Year-by-year backtest detail
short(num(run(q4["forecast_backtest_detail"])))

### Chart 8: Backtest error by method and category (lower is better)

In [ ]:
bt_cols = [("M1 Linear", "mape_m1_linear"), ("M2 Log-linear", "mape_m2_log"),
           ("M3 Flat MA", "mape_m3_flat_ma"), ("M4 MA + drift", "mape_m4_ma_drift"),
           ("Holt-Winters", "mape_hw")]
cats_order = [v[0] for v in CATS.values()]
btp = bt.set_index("kind_of_business").loc[cats_order]
x = np.arange(len(cats_order)); w = 0.16

fig, ax = plt.subplots(figsize=(11, 5))
for i, (name, col) in enumerate(bt_cols):
    vals = btp[col].values
    ax.bar(x + (i - 2) * w, vals, width=w - 0.02, color=METHOD_COLORS[name], label=name)
for j, cat in enumerate(cats_order):
    best = btp.loc[cat, [c for _, c in bt_cols]].astype(float)
    i_best = int(np.argmin(best.values))
    ax.annotate("best", (x[j] + (i_best - 2) * w, best.values[i_best]), xytext=(0, 3),
                textcoords="offset points", ha="center", fontsize=8, color=INK)
ax.set_xticks(x, cats_order)
ax.set_ylabel("MAPE, 2017-2019 (%)")
ax.set_title("Which forecast method would have worked? Backtest 2017-2019", loc="left", fontsize=13)
ax.legend(frameon=False, ncol=5, loc="upper left", fontsize=8.5)
ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig("figures/08_backtest_mape.png", dpi=150, bbox_inches="tight")
plt.show()

### Sensitivity: how much does the fit window change the Department stores forecast?

In [ ]:
sens = num(run(q4["forecast_sensitivity"]))
sens

### Chart 9: 2023 forecast by fit window

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2))
ypos = np.arange(len(sens))[::-1]
ax.barh(ypos, sens.forecast_2023_linear / 1000, color=METHOD_COLORS["M1 Linear"], height=0.55,
        label="M1 linear")
ax.scatter(sens.forecast_2023_log / 1000, ypos, color=METHOD_COLORS["M2 Log-linear"], zorder=5,
           s=45, label="M2 log-linear", edgecolor="white", linewidth=1.2)
for yp, v in zip(ypos, sens.forecast_2023_linear / 1000):
    ax.text(v - 2, yp, f"{v:,.0f}", va="center", ha="right", fontsize=9, color="white", fontweight="bold")
ax.axvline(a20, color=INK, lw=1, ls="--")
ax.text(a20 + 1, ypos[-1] - 0.55, f"2020 actual {a20:,.0f}", fontsize=8.5, color=INK)
ax.set_yticks(ypos, sens.window_name)
ax.set_xlabel("Forecast 2023 annual sales ($B)")
ax.set_xlim(0, 165)
ax.set_ylim(-0.9, len(sens) - 0.5)
ax.set_title("Department stores: 2023 forecast by fit window", loc="left", fontsize=13)
ax.legend(frameon=False, loc="lower right", fontsize=8.5)
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig("figures/09_forecast_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()

### Chart 10: Forecasts for all five categories
Category color = actual. Shaded = M1 95% interval. Dashed = M1, dotted = M2 log-linear, purple = Holt-Winters.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(10, 14))
for ax, (cat, (label, color)) in zip(axes, CATS.items()):
    h = yearly[(yearly.kind_of_business == cat) & (yearly.sales_year >= 2005)]
    f = fa[fa.kind_of_business == label].sort_values("forecast_year")
    ax.plot(h.sales_year, h.year_sales / 1000, color=color, lw=2.2, marker="o", ms=3.5)
    ax.fill_between(f.forecast_year, f.m1_lower_95 / 1000, f.m1_upper_95 / 1000, color=INK2, alpha=0.12, lw=0)
    ax.plot(f.forecast_year, f.m1_linear / 1000, color=INK, lw=1.5, ls="--", label="M1 linear")
    ax.plot(f.forecast_year, f.m2_log_linear / 1000, color=INK, lw=1.5, ls=":", label="M2 log-linear")
    ax.plot(f.forecast_year, f.hw / 1000, color=METHOD_COLORS["Holt-Winters"], lw=1.5, label="Holt-Winters")
    last = f.iloc[-1]
    ax.set_title(f"{label}: 2023 M1 {last.m1_linear/1000:,.1f}B, M2 {last.m2_log_linear/1000:,.1f}B, "
                 f"HW {last.hw/1000:,.1f}B", loc="left")
    ax.set_ylabel("$B / year")
    ax.set_xlim(2004.5, 2023.5)
    ax.set_xticks(range(2005, 2024, 2))
axes[0].legend(frameon=False, loc="upper left", fontsize=8.5)
fig.suptitle("Annual sales and 2020-2023 forecasts", x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK)
fig.tight_layout()
fig.savefig("figures/10_all_category_forecasts.png", dpi=150, bbox_inches="tight")
plt.show()

### Step 4 findings

**Department stores forecast** (annual sales, $B; 2020 actual = 113.1)

| Method | 2021 | 2022 | 2023 |
|---|---|---|---|
| M1 Linear trend | 124.3 | 118.5 | 112.7 |
| M1 95% interval | 116.5 to 132.1 | 110.3 to 126.7 | 104.1 to 121.3 |
| M2 Log-linear | 127.3 | 122.8 | 118.4 |
| M3 Flat moving average | 135.0 | 135.0 | 135.0 |
| M4 Moving average + drift | 125.4 | 120.6 | 115.7 |
| M5 Rebased (no recovery) | 107.3 | 101.5 | 95.7 |
| Holt-Winters | 125.3 | 122.2 | 120.0 |

- **Central estimate for 2023: about $113B to $120B** (M1, M2, M4 and Holt-Winters agree within $7.3B), down 11% to 17% from 2019.
- **Downside: $95.7B** if 2020's drop is permanent (M5).
- **M3 is not credible**: holding the 2019 level flat contradicts 19 years of decline and had the worst backtest error (8.7%).
- **2020 fell outside the M1 interval**: actual $113.1B vs trend $130.2B (-13.1%; interval $122.7B to $137.6B). COVID is a structural break, not normal year-to-year noise.
- The trend loses about $5.8B a year (-3.6% a year), R² = 0.98 on 2010-2019.

**Backtest** (fit 2010-2016, predict 2017-2019; MAPE %)

| Category | M1 | M2 | M3 | M4 | HW | Best |
|---|---|---|---|---|---|---|
| Department stores | 3.7 | 4.8 | 8.7 | **2.5** | 7.4 | M4 |
| E-commerce | 12.2 | 5.3 | 20.7 | 4.7 | **4.5** | HW |
| Warehouse clubs | **0.4** | 1.6 | 6.6 | 3.2 | 3.6 | M1 |
| Book stores | 9.4 | **4.2** | 12.2 | 6.0 | 7.8 | M2 |
| Gasoline stations | 7.6 | 8.1 | 13.2 | 36.7 | **5.6** | HW |

- No single best method. Steady linear series suit M1 (warehouse clubs); compounding series suit M2 or HW (e-commerce, book stores).
- M4 (drift) fails on gasoline (36.7%): it extends short-term price swings.
- Flat (M3) is never best: every category has a trend.

**Fit window sensitivity** (Department stores, 2023)

| Window | Slope ($B/yr) | R² | 2023 linear | 2023 log-linear |
|---|---|---|---|---|
| 1992-2019 | -2.5 | 0.53 | 149.1 | 149.6 |
| 2001-2019 | -5.2 | 0.98 | 118.4 | 126.3 |
| 2010-2019 (default) | -5.8 | 0.98 | 112.7 | 118.4 |
| 2015-2019 | -6.9 | 0.96 | 106.2 | 111.5 |
| 2010-2020 (incl. COVID) | -6.6 | 0.95 | 105.0 | 110.4 |

- The decline is **accelerating**: every more recent window gives a steeper slope.
- The full 1992-2019 window is a poor fit (R² 0.53) because it mixes the growth years with the decline.
- "Linear trend reaches zero in 2038-2045" is an artifact of straight-line math, not a prediction; the log-linear model never reaches zero.

**Limitations**: only 10 annual points in the fit; the interval assumes past errors continue; nominal dollars; the 2020 structural break makes every method less reliable.

## Step 5: Extended analysis

Ten further angles on the same data, all computed in SQL (`sql/05_extended.sql`):

| # | Question | Technique |
|---|---|---|
| 5.1 | How much of total retail does each category hold, and who gained share? | join to the market total, `LAG` for share change |
| 5.2 | When did e-commerce overtake department stores? | conditional aggregation, `LAG` crossover flag |
| 5.3 | Is the holiday season getting more or less important? | `REGR_SLOPE` of Nov/Dec share on year |
| 5.4 | How deep was each recession, and did sales recover? | running peak with `MAX() OVER`, drawdowns |
| 5.5 | How did COVID play out month by month? | 2020 YoY by phase |
| 5.6 | Where do the five rank among all retail categories in 2020? | `RANK()` over every industry category |
| 5.7 | Where do they rank on 27-year growth? | CAGR, `PERCENT_RANK()` |
| 5.8 | Do categories move together, and how cyclical is each? | `CORR`, market beta via `REGR_SLOPE` |
| 5.9 | When was each category most volatile? | rolling 36-month `STDDEV` |
| 5.10 | Do extra weekend days distort month-to-month growth? | calendar built with `generate_series`, `CORR` |

In [ ]:
q5 = load_queries("05_extended.sql")
for view in ["create_view_drawdown", "create_view_calendar"]:
    execute(q5[view])


def label_ends(ax, points, min_gap, dx=0.4):
    """Direct-label line ends at (x, y, text), nudging labels apart vertically by min_gap."""
    placed = []
    for x, y, txt in sorted(points, key=lambda p: p[1]):
        y_adj = max(y, placed[-1] + min_gap) if placed else y
        placed.append(y_adj)
        ax.annotate(txt, (x, y), xytext=(x + dx, y_adj), va="center", fontsize=9, color=INK,
                    arrowprops=dict(arrowstyle="-", color=GRID, lw=0.8))

### 5.1 Share of total retail

In [ ]:
share = short(num(run(q5["share_of_retail"])))
short(num(run(q5["share_summary"])))

In [ ]:
short(num(run(q5["market_growth_vs_category"])), "series")

#### Chart 11: Share of total U.S. retail sales

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for cat, (label, color) in CATS.items():
    d = share[share.kind_of_business == label]
    ax.plot(d.sales_year, d.pct_of_retail, color=color, lw=2.2, label=label)
    ax.annotate(f"{label} {d.pct_of_retail.iloc[-1]:.1f}%", (d.sales_year.iloc[-1], d.pct_of_retail.iloc[-1]),
                xytext=(6, 0), textcoords="offset points", va="center", fontsize=9, color=INK)
ax.set_xlim(1991.5, 2025.5)
ax.set_xticks(range(1992, 2021, 4))
ax.set_ylabel("% of total retail sales")
ax.set_title("Share of total U.S. retail sales, 1992-2020", loc="left", fontsize=13)
ax.legend(frameon=False, loc="upper left", fontsize=8.5)
fig.tight_layout()
fig.savefig("figures/11_share_of_retail.png", dpi=150, bbox_inches="tight")
plt.show()

#### Findings

- **E-commerce**: 1.9% of retail (1992) to 15.2% (2020), +13.2 points. Share gains sped up: about +0.1 to +0.5 points a year before 2014 (2009: +0.7), +0.7 to +0.9 in 2015-2018, +1.3 in 2019, +2.2 in 2020.
- **Warehouse clubs**: 2.2% to a 9.8% peak in 2009, then flat at 9.1% to 9.6%. Share growth stopped when the PELT slowdown began.
- **Department stores**: 9.8% to 2.0%, -7.8 points; lost share every year except 2009.
- **Gasoline**: between 7.4% and 13.0%, moving with oil prices, with no long-run gain.
- **Book stores**: 0.46% to 0.11%.
- Growth vs the market (total retail 4.2%/yr): e-commerce 11.7%, warehouse clubs 9.8% and gasoline 4.4% beat it; book stores 0.2% and department stores -1.0% trailed it.

### 5.2 E-commerce vs department stores

In [ ]:
ecd = num(run(q5["ecom_vs_dept_annual"]))
ecd

In [ ]:
run(q5["ecom_vs_dept_monthly_firsts"])

#### Chart 12: The crossover

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ecd.sales_year, ecd.ecom / 1000, color=CATS["Electronic shopping and mail-order houses"][1], lw=2.4,
        marker="o", ms=3.5, label="E-commerce")
ax.plot(ecd.sales_year, ecd.dept / 1000, color=CATS["Department stores"][1], lw=2.4, marker="o", ms=3.5,
        label="Department stores")
xo = ecd[ecd.event == "crossover"].iloc[0]
ax.annotate(f"Crossover {int(xo.sales_year)}\nE-com {xo.ecom/1000:,.0f}B vs Dept {xo.dept/1000:,.0f}B",
            (xo.sales_year, xo.ecom / 1000), xytext=(1996, 420), fontsize=9, color=INK,
            arrowprops=dict(arrowstyle="->", color=INK2, lw=0.9))
last = ecd.iloc[-1]
ax.annotate(f"{last.ecom_to_dept_ratio:.1f}x in 2020", (last.sales_year, last.ecom / 1000),
            xytext=(-70, 0), textcoords="offset points", fontsize=9, color=INK, va="center")
ax.set_ylabel("Annual sales ($B)")
ax.set_title("E-commerce vs department stores, 1992-2020", loc="left", fontsize=13)
ax.legend(frameon=False, loc="upper left", fontsize=9)
fig.tight_layout()
fig.savefig("figures/12_ecom_vs_dept.png", dpi=150, bbox_inches="tight")
plt.show()

#### Findings

| Milestone | Date |
|---|---|
| E-commerce as a share of department stores, 1992 | 0.20x |
| First month e-commerce > department stores | Jan 2006 |
| First full year e-commerce > department stores | 2007 ($223.7B vs $209.4B) |
| Last month department stores > e-commerce | Dec 2008 |
| First December e-commerce > department stores | Dec 2009 |
| Ratio in 2020 | 7.5x |

- The crossover took about three years: first in low months (Jan 2006), last in department stores' strongest month (December, 2009).
- The gap widened from $14B (2007) to $736B (2020).

### 5.3 Holiday concentration over time

In [ ]:
hol = short(num(run(q5["holiday_share_by_year"])))
short(num(run(q5["holiday_share_trend"])))

In [ ]:
amp = short(num(run(q5["seasonal_amplitude"])))
amp.pivot(index="sales_year", columns="kind_of_business", values="peak_to_trough_ratio").iloc[::4]

#### Chart 13: December share of annual sales

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for cat, (label, color) in CATS.items():
    d = hol[hol.kind_of_business == label]
    ax.plot(d.sales_year, d.dec_pct, color=color, lw=2, label=label)
label_ends(ax, [(2020, hol[hol.kind_of_business == v[0]].dec_pct.iloc[-1], v[0]) for v in CATS.values()], 0.45)
ax.axhline(100 / 12, color=INK2, lw=1, ls="--")
ax.text(2021.2, 100 / 12 - 0.35, "flat year 8.3%", fontsize=8.5, color=INK2)
ax.set_xlim(1991.5, 2024.5)
ax.set_xticks(range(1992, 2021, 4))
ax.set_ylabel("December as % of year")
ax.set_title("December share of annual sales", loc="left", fontsize=13)
ax.legend(frameon=False, loc="lower left", fontsize=8.5, ncol=3)
fig.tight_layout()
fig.savefig("figures/13_december_share.png", dpi=150, bbox_inches="tight")
plt.show()

#### Findings

| Category | Dec share 1992-96 | Dec share 2015-19 | Trend (points per decade) | R² |
|---|---|---|---|---|
| Warehouse clubs | 13.5% | 10.4% | -1.36 | 0.84 |
| Book stores | 13.7% | 11.6% | -1.11 | 0.73 |
| Department stores | 15.1% | 13.8% | -0.53 | 0.79 |
| Gasoline stations | 8.6% | 7.9% | -0.34 | 0.13 |
| E-commerce | 13.0% | 12.2% | -0.20 | 0.07 |

- December's share fell in all five; the trend is strong for warehouse, book and department stores.
- November's share fell too (slopes -0.05 to -0.55 points per decade), so spending did not move from December into November at the category level.
- Years got flatter: peak-to-trough month ratio for warehouse clubs 2.23 (1992) to 1.43 (2019); department stores 2.64 to 2.13; e-commerce 1.98 to 1.77.
- E-commerce's December share barely moved on this measure, but STL (5.11) shows its December bump shrinking. Fast growth inflates late-year months in a share-of-year measure.
- *Hypotheses*: warehouse clubs' mix shifted toward everyday groceries; gift cards move spending into January.

### 5.4 Drawdowns and recessions

In [ ]:
short(num(run(q5["max_drawdown"])))

In [ ]:
rec = short(num(run(q5["recession_impact"])))
rec

#### Chart 14: Drawdown from prior peak (12-month MA)

In [ ]:
dd = short(num(run("SELECT * FROM v_drawdown ORDER BY kind_of_business, sales_month")))
dd["sales_month"] = pd.to_datetime(dd["sales_month"])
fig, ax = plt.subplots(figsize=(11, 5.2))
for s0, s1 in [("2001-03-01", "2001-11-01"), ("2007-12-01", "2009-06-01"), ("2020-02-01", "2020-04-01")]:
    ax.axvspan(pd.Timestamp(s0), pd.Timestamp(s1), color=GRID, alpha=0.6, lw=0)
for cat, (label, color) in CATS.items():
    d = dd[dd.kind_of_business == label]
    ax.plot(d.sales_month, d.drawdown_pct, color=color, lw=2, label=label)
ax.axhline(0, color=INK2, lw=0.8)
ax.text(pd.Timestamp("2008-02-01"), -66, "Great\nRecession", fontsize=8, color=INK2)
ax.text(pd.Timestamp("2001-04-01"), -66, "2001", fontsize=8, color=INK2)
ax.text(pd.Timestamp("2019-01-01"), -66, "COVID", fontsize=8, color=INK2)
ax.set_ylim(-70, 3)
ax.set_ylabel("% below prior peak")
ax.set_title("Drawdown from prior peak, 12-month moving average (gray = NBER recessions)", loc="left", fontsize=13)
ax.legend(frameon=False, loc="lower left", fontsize=8.5)
fig.tight_layout()
fig.savefig("figures/14_drawdowns.png", dpi=150, bbox_inches="tight")
plt.show()

#### Chart 15: Peak-to-trough decline by recession

In [ ]:
rec_order = ["2001 dot-com", "2008-09 Great Recession", "2020 COVID-19"]
x = np.arange(len(rec_order)); w = 0.16
fig, ax = plt.subplots(figsize=(10, 4.8))
for i, (cat, (label, color)) in enumerate(CATS.items()):
    vals = [rec[(rec.recession == r) & (rec.kind_of_business == label)].start_to_trough_pct.iloc[0] for r in rec_order]
    ax.bar(x + (i - 2) * w, vals, width=w - 0.02, color=color, label=label)
    for xi, v in zip(x, vals):
        if v == 0:
            ax.text(xi + (i - 2) * w, -0.4, "no\ndecline", ha="center", va="top", fontsize=7, color=INK2)
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xticks(x, rec_order)
ax.set_ylabel("12-mo MA: recession start to trough (%)")
ax.set_title("Recession impact by category", loc="left", fontsize=13)
ax.legend(frameon=False, loc="lower left", fontsize=8.5, ncol=2)
ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig("figures/15_recession_impact.png", dpi=150, bbox_inches="tight")
plt.show()

#### Findings

| Recession | Category | Start to trough | Status |
|---|---|---|---|
| 2001 | Department stores | -7.6% | never regained start level |
| 2001 | Gasoline stations | -4.3% | recovered in 22 months |
| 2001 | Book stores | -1.1% | recovered in 5 months |
| 2001 | E-commerce | -0.7% | recovered in 11 months |
| 2008-09 | Gasoline stations | -16.4% | recovered in 37 months |
| 2008-09 | Department stores | -11.9% | never regained start level |
| 2008-09 | Book stores | -11.3% | never regained start level |
| 2020 | Book stores | -27.0% | still falling at data end |
| 2020 | Department stores | -16.1% | still falling at data end |
| 2020 | Gasoline stations | -15.6% | still falling at data end |

- **Recession-proof**: warehouse clubs had no decline in any recession; e-commerce had none in 2008-09 or 2020. Their worst-ever drawdowns: -0.5% and -2.3%.
- **Cyclical**: gasoline falls hard and recovers (max drawdown -28.2%, Oct 2009).
- **Ratchet**: department and book stores never recover a recession's losses; each recession resets them lower. Max drawdowns -51.4% and -63.2% (Dec 2020).
- COVID differs from 2001 and 2008-09: it was an in-person shopping shock, not a credit or income shock, so store-based categories fell and online and bulk-buying categories grew.

### 5.5 COVID-19 month by month

In [ ]:
cov = num(run(q5["covid_monthly_yoy"]))
cov

In [ ]:
short(num(run(q5["covid_phases"])))

#### Chart 16: 2020 year-over-year change by month

In [ ]:
cov["sales_month"] = pd.to_datetime(cov["sales_month"])
col_map = {"ecommerce": "Electronic shopping and mail-order houses", "warehouse": "Warehouse clubs and superstores",
           "gasoline": "Gasoline stations", "department": "Department stores", "books": "Book stores"}
fig, ax = plt.subplots(figsize=(10, 5.2))
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-05-01"), color=GRID, alpha=0.6, lw=0)
ax.set_ylim(-80, 38)
ax.text(pd.Timestamp("2020-03-05"), 33, "lockdown", fontsize=8.5, color=INK2)
for col, cat in col_map.items():
    label, color = CATS[cat]
    ax.plot(cov.sales_month, cov[col], color=color, lw=2.2, marker="o", ms=4, label=label)
    ax.annotate(label, (cov.sales_month.iloc[-1], cov[col].iloc[-1]), xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=8.5, color=INK)
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xlim(pd.Timestamp("2019-12-15"), pd.Timestamp("2021-02-20"))
ax.set_xticks(pd.date_range("2020-01-01", "2020-12-01", freq="MS"))
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%b"))
ax.set_ylabel("YoY change (%)")
ax.set_title("2020 sales vs same month of 2019", loc="left", fontsize=13)
fig.tight_layout()
fig.savefig("figures/16_covid_monthly.png", dpi=150, bbox_inches="tight")
plt.show()

#### Findings

| Category | Jan-Feb | Lockdown (Mar-May) | Reopening (Jun-Dec) | Worst month |
|---|---|---|---|---|
| E-commerce | +10.4% | +23.6% | +22.0% | Jan +9.1% |
| Warehouse clubs | +6.2% | +9.2% | +7.2% | Apr +3.3% |
| Gasoline stations | +7.5% | -29.3% | -13.4% | Apr -39.1% |
| Department stores | -1.0% | -31.0% | -12.9% | Apr -44.5% |
| Book stores | -2.0% | -55.6% | -24.9% | Apr -74.2% |

- April 2020 was the low point for every store category.
- Warehouse clubs peaked in March (+14.4%): stock-up buying before lockdowns.
- No store category was back to 2019 levels by December.
- Department stores got worse into the holidays (Oct -6.9%, Nov -16.2%, Dec -20.5%), while e-commerce ran +19% to +28% from March to November.
- E-commerce's weakest pandemic month, December (+12.9%), is a calendar effect: Cyber Monday fell on Nov 30 in 2020 vs Dec 2 in 2019 (November was +25.0%).

### 5.6 COVID winners and losers across all industry categories

In [ ]:
allcov = num(run(q5["covid_all_categories"]))
allcov["kind_of_business"] = allcov["kind_of_business"].replace(SHORT)
print(len(allcov), "industry categories with complete 2019 and 2020 data")
allcov

#### Chart 17: 2020 vs 2019, every industry category (the five analysis categories highlighted)

In [ ]:
a = allcov.sort_values("change_pct")
colors = [CATS[k][1] if k in CATS else "#c9c8c2"
          for k in a["kind_of_business"].map({v[0]: k for k, v in CATS.items()}).fillna("")]
fig, ax = plt.subplots(figsize=(9, 0.26 * len(a) + 1.2))
ax.barh(range(len(a)), a.change_pct, color=colors, height=0.75)
ax.set_yticks(range(len(a)), a.kind_of_business, fontsize=8)
for i, (v, hl) in enumerate(zip(a.change_pct, a.in_analysis)):
    ax.text(v + (0.8 if v >= 0 else -0.8), i, f"{v:+.1f}%", va="center", ha="left" if v >= 0 else "right",
            fontsize=7.5, color=INK, fontweight="bold" if hl else "normal")
ax.axvline(0, color=INK2, lw=0.8)
ax.set_xlabel("2020 vs 2019 (%)")
ax.set_title("COVID winners and losers: 2020 vs 2019", loc="left", fontsize=13)
ax.grid(axis="y", visible=False)
ax.set_xlim(a.change_pct.min() - 10, a.change_pct.max() + 10)
ax.set_ylim(-0.7, len(a) - 0.3)
fig.tight_layout()
fig.savefig("figures/17_covid_all_categories.png", dpi=150, bbox_inches="tight")
plt.show()

#### Findings

- 51 industry categories had complete 2019 and 2020 data.
- **Top**: sporting goods +21.8%, e-commerce +20.4%, hardware +19.7%, nonstore retailers +17.6%, household appliances +14.0%, beer/wine/liquor +13.6%, grocery +12.6% to +13.3%.
- **Bottom**: department stores excl. discount -48.6%, women's clothing -35.1%, gift and souvenir -30.6%, clothing -28.1%, family clothing -27.5%, book stores -27.4%.
- **The five**: e-commerce #2, warehouse clubs #15, gasoline #35, department stores #37, book stores #46.
- Winners share a theme: home-centered spending (home improvement, food and drink at home, outdoor recreation) and online delivery. Losers: apparel, gifts and browsing-based stores.
- Inside department stores: discount stores -1.6% vs traditional department stores -48.6%.

### 5.7 Long-run growth leaderboard (CAGR 1992-2019)

In [ ]:
lead = num(run(q5["growth_leaderboard"]))
lead["kind_of_business"] = lead["kind_of_business"].replace(SHORT)
lead

#### Chart 18: 27-year growth rate, every industry category

In [ ]:
a = lead.sort_values("cagr_pct")
colors = [CATS[k][1] if k in CATS else "#c9c8c2"
          for k in a["kind_of_business"].map({v[0]: k for k, v in CATS.items()}).fillna("")]
fig, ax = plt.subplots(figsize=(9, 0.26 * len(a) + 1.2))
ax.barh(range(len(a)), a.cagr_pct, color=colors, height=0.75)
ax.set_yticks(range(len(a)), a.kind_of_business, fontsize=8)
for i, (v, hl) in enumerate(zip(a.cagr_pct, a.in_analysis)):
    ax.text(v + (0.15 if v >= 0 else -0.15), i, f"{v:.1f}%", va="center", ha="left" if v >= 0 else "right",
            fontsize=7.5, color=INK, fontweight="bold" if hl else "normal")
ax.axvline(0, color=INK2, lw=0.8)
mkt = 100 * ((share.groupby("sales_year").total_sales.first()[2019]
              / share.groupby("sales_year").total_sales.first()[1992]) ** (1 / 27) - 1)
ax.axvline(mkt, color=INK, lw=1, ls="--", zorder=0)
ax.text(mkt + 0.1, -1.2, f"total retail {mkt:.1f}%", fontsize=8, color=INK, va="top")
ax.set_ylim(-1.8, len(a) - 0.3)
ax.set_xlim(a.cagr_pct.min() - 1.2, a.cagr_pct.max() + 1.2)
ax.set_xlabel("Compound annual growth rate, 1992-2019 (%)")
ax.set_title("Long-run growth by category", loc="left", fontsize=13)
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig("figures/18_growth_leaderboard.png", dpi=150, bbox_inches="tight")
plt.show()

#### Findings

- 51 industry categories, CAGR 1992-2019; the total retail benchmark is 4.2%.
- **The five span the whole list**: e-commerce #1 (11.7%), warehouse clubs #2 (9.8%), gasoline #14 (4.4%), book stores #47 (0.2%), department stores #50 (-1.0%).
- The top of the list is formats: nonstore retailers #3 (9.0%) and other general merchandise stores #4 (8.1%, which includes warehouse clubs).
- The bottom is traditional store formats and products that went digital: department stores excl. discount #51 (-2.6%), men's clothing #49 (-0.9%), discount department stores #48 (0.0%), office supplies and stationery #46 (0.7%).

### 5.8 Co-movement and market beta

In [ ]:
corr = short(short(num(run(q5["yoy_correlation_matrix"])), "category_a"), "category_b")
cm = corr.pivot(index="category_a", columns="category_b", values="corr_yoy").loc[cats_order, cats_order]
cm

In [ ]:
beta = short(num(run(q5["market_beta"])))
beta

#### Chart 19: Correlation of monthly YoY growth, 1993-2019

In [ ]:
cmap_div = LinearSegmentedColormap.from_list("div2", ["#2a78d6", "#f0efec", "#e34948"])
fig, ax = plt.subplots(figsize=(7.2, 5.6))
im = ax.imshow(cm.values, cmap=cmap_div, vmin=-1, vmax=1)
ax.set_xticks(range(5), cats_order, rotation=30, ha="right"); ax.set_yticks(range(5), cats_order)
ax.grid(False)
for i in range(5):
    for j in range(5):
        v = cm.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9.5,
                color="white" if abs(v) > 0.6 else INK)
for sp in ax.spines.values(): sp.set_visible(False)
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
ax.set_title("Do categories grow together?", loc="left", fontsize=13)
fig.tight_layout()
fig.savefig("figures/19_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

#### Chart 20: Market beta (sensitivity to total retail growth)

In [ ]:
b = beta.set_index("kind_of_business").loc[cats_order]
x = np.arange(5); w = 0.36
fig, ax = plt.subplots(figsize=(9, 4.4))
ax.bar(x - w / 2, b.beta_vs_total, width=w - 0.03, color="#2a78d6", label="vs total retail")
ax.bar(x + w / 2, b.beta_vs_ex_gas, width=w - 0.03, color="#eb6834", label="vs total retail excl. gasoline")
for xi, v1, v2 in zip(x, b.beta_vs_total, b.beta_vs_ex_gas):
    ax.text(xi - w / 2, v1 + 0.03, f"{v1:.2f}", ha="center", fontsize=8.5, color=INK)
    ax.text(xi + w / 2, v2 + 0.03, f"{v2:.2f}", ha="center", fontsize=8.5, color=INK)
ax.axhline(1, color=INK2, lw=1, ls="--")
ax.text(4.45, 1.03, "beta = 1", fontsize=8, color=INK2, ha="right")
ax.set_xticks(x, cats_order)
ax.set_ylabel("Beta (YoY on market YoY)")
ax.set_title("How strongly each category amplifies market swings, 1993-2019", loc="left", fontsize=13)
ax.legend(frameon=False, loc="upper right", fontsize=8.5)
ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig("figures/20_market_beta.png", dpi=150, bbox_inches="tight")
plt.show()

#### Findings

- All pairwise YoY correlations are positive (0.14 to 0.68); the strongest are among store-based categories: warehouse clubs and department stores 0.68, warehouse clubs and book stores 0.60, department and book stores 0.55.
- Gasoline correlates least with everything (0.14 to 0.33): its driver is fuel prices, not shopping.
- **No month-to-month substitution**: e-commerce and department stores correlate +0.35. They rise and fall together in the short run; substitution shows up only in long-run levels (5.2).

| Category | Beta vs total retail | Beta vs retail excl. gasoline |
|---|---|---|
| Gasoline stations | 1.91 | 1.23 |
| E-commerce | 1.00 | 1.11 |
| Warehouse clubs | 0.78 | 0.83 |
| Book stores | 0.72 | 0.83 |
| Department stores | 0.61 | 0.71 |

- Gasoline's 1.91 is partly mechanical (gasoline is inside the total); 1.23 without it.
- Department stores have the lowest beta: their decline is structural, not cyclical.

### 5.9 Volatility regimes

In [ ]:
short(num(run(q5["volatility_by_era"])))

#### Chart 21: Rolling 36-month volatility of YoY growth

In [ ]:
vol = short(num(run(q5["rolling_volatility"])))
vol["sales_month"] = pd.to_datetime(vol["sales_month"])
fig, ax = plt.subplots(figsize=(11, 5))
for cat, (label, color) in CATS.items():
    d = vol[(vol.kind_of_business == label) & vol.yoy_sd_36m.notna()]
    ax.plot(d.sales_month, d.yoy_sd_36m, color=color, lw=2, label=label)
ax.set_ylabel("SD of YoY growth, trailing 36 months (pp)")
ax.set_title("Volatility of growth over time", loc="left", fontsize=13)
ax.legend(frameon=False, loc="upper left", fontsize=8.5)
fig.tight_layout()
fig.savefig("figures/21_rolling_volatility.png", dpi=150, bbox_inches="tight")
plt.show()

#### Findings

| Category | 1993-2000 | 2001-2007 | 2008-2010 | 2011-2019 | 2020 |
|---|---|---|---|---|---|
| Gasoline stations | 7.1 | 10.6 | 23.6 | 10.9 | 13.2 |
| Book stores | 5.7 | 5.9 | 5.8 | 5.8 | 21.0 |
| E-commerce | 6.0 | 6.2 | 6.9 | 4.2 | 6.2 |
| Department stores | 3.0 | 2.6 | 3.4 | 3.1 | 12.2 |
| Warehouse clubs | 5.0 | 4.0 | 4.0 | 2.3 | 3.4 |

*(standard deviation of monthly YoY growth, percentage points)*

- Gasoline is structurally volatile, peaking in 2008-2012 (rolling SD about 24 points) with the oil price swings.
- E-commerce and warehouse clubs got steadier as they grew.
- Book and department stores were stable until 2020, when volatility jumped 3-4x: COVID was their first large shock.

### 5.10 Calendar (trading-day) effects
The data is **not** trading-day adjusted, so a month with one more Saturday than last year can show growth that has nothing to do with demand.

In [ ]:
short(num(run(q5["calendar_effect"])))

#### Chart 22: Average YoY residual by weekend days gained or lost vs last year

In [ ]:
calx = short(num(run(q5["calendar_by_weekend_delta"])))
fig, ax = plt.subplots(figsize=(9, 5))
for cat, (label, color) in CATS.items():
    d = calx[calx.kind_of_business == label]
    ax.plot(d.weekend_days_vs_ly, d.avg_yoy_resid_pp, color=color, lw=2.2, marker="o", ms=6, label=label)
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xticks([-2, -1, 0, 1, 2], ["-2", "-1", "0", "+1", "+2"])
ax.set_xlabel("Weekend days vs same month last year (n = 10, 83, 138, 83, 10 months)")
ax.set_ylabel("Avg YoY residual (pp)")
ax.set_title("YoY growth residual by weekend days gained or lost vs last year", loc="left", fontsize=13)
ax.legend(frameon=False, loc="upper right", fontsize=8.5)
fig.tight_layout()
fig.savefig("figures/22_calendar_effect.png", dpi=150, bbox_inches="tight")
plt.show()

#### Findings

| Category | Correlation | YoY points per extra weekend day |
|---|---|---|
| E-commerce | -0.58 | -2.70 |
| Book stores | -0.20 | -1.09 |
| Gasoline stations | -0.09 | -0.65 |
| Department stores | +0.16 | +0.43 |
| Warehouse clubs | +0.24 | +0.63 |

- **E-commerce loses on weekends**: with 2 fewer weekend days than last year its YoY runs +5.8 points above trend; with 2 more, -3.5 points. That is a 9-point swing from the calendar alone.
- **Warehouse and department stores gain on weekends** (store trips).
- *Hypothesis*: online and mail-order purchases concentrate on weekdays (work hours, business buyers); store trips concentrate on weekends.
- Holiday timing matters as well: Thanksgiving on Nov 28, 2019 put Cyber Monday in December and produced e-commerce's Nov 2019 (+4.9%) and Dec 2019 (+26.7%) readings.
- Implication: any single month's YoY needs a calendar check before it is read as a change in demand.

### 5.11 Seasonal decomposition (STL)
STL splits Department stores' monthly sales into trend, seasonal and residual components, independent of the SQL moving averages. Multiplicative decomposition via log sales, so the seasonal component is a % effect.

In [ ]:
from statsmodels.tsa.seasonal import STL

stl_res = {}
for cat, (label, _) in CATS.items():
    s = monthly[monthly.kind_of_business == cat].set_index("sales_month")["sales"].asfreq("MS")
    stl_res[label] = STL(np.log(s), period=12, robust=True).fit()

r = stl_res[DEPT]
seasonal_pct = (np.exp(r.seasonal) - 1) * 100
dec_effect = seasonal_pct[seasonal_pct.index.month == 12]
print("Department stores December seasonal effect, first vs last year:",
      f"{dec_effect.iloc[0]:+.1f}% -> {dec_effect.iloc[-1]:+.1f}%")

#### Chart 23: STL decomposition, Department stores

In [ ]:
s = monthly[monthly.kind_of_business == DEPT].set_index("sales_month")["sales"]
fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
axes[0].plot(s.index, s / 1000, color=CATS[DEPT][1], lw=1)
axes[0].set_title("Observed ($B / month)", loc="left")
axes[1].plot(r.trend.index, np.exp(r.trend) / 1000, color=CATS[DEPT][1], lw=2)
axes[1].set_title("Trend ($B / month)", loc="left")
axes[2].plot(seasonal_pct.index, seasonal_pct, color=CATS[DEPT][1], lw=1)
axes[2].axhline(0, color=INK2, lw=0.8)
axes[2].set_title("Seasonal effect (% vs trend)", loc="left")
axes[3].plot(r.resid.index, (np.exp(r.resid) - 1) * 100, color=INK2, lw=0.8)
axes[3].axhline(0, color=INK2, lw=0.8)
axes[3].set_title("Residual (%)", loc="left")
fig.suptitle("Department stores: STL decomposition", x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK)
fig.tight_layout()
fig.savefig("figures/23_stl_department.png", dpi=150, bbox_inches="tight")
plt.show()

#### Chart 24: December seasonal effect over time, all five (STL)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for cat, (label, color) in CATS.items():
    sp = (np.exp(stl_res[label].seasonal) - 1) * 100
    dec = sp[sp.index.month == 12]
    ax.plot(dec.index.year, dec.values, color=color, lw=2.2, label=label)
    ax.annotate(label, (dec.index.year[-1], dec.values[-1]), xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=8.5, color=INK)
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xlim(1991.5, 2025.5)
ax.set_xticks(range(1992, 2021, 4))
ax.set_ylabel("December effect vs trend (%)")
ax.set_title("December seasonal effect vs trend (STL seasonal component)", loc="left", fontsize=13)
fig.tight_layout()
fig.savefig("figures/24_stl_december_effect.png", dpi=150, bbox_inches="tight")
plt.show()

#### Findings

| Category | December effect vs trend, 1992 | 2019 | 2020 |
|---|---|---|---|
| Department stores | +86.4% | +67.2% | +64.8% |
| Book stores | +67.0% | +52.6% | +54.9% |
| Warehouse clubs | +65.5% | +19.0% | +17.2% |
| E-commerce | +50.4% | +31.7% | +27.1% |
| Gasoline stations | +1.3% | +0.6% | +2.8% |

- STL confirms 5.3: the December bump shrank in every holiday category; warehouse clubs lost most of theirs.
- STL also shows the e-commerce bump shrinking, which the share-of-year measure missed.
- The largest residual in 29 years is April 2020 for every store category: book stores -74.6%, department stores -42.3%, gasoline -32.9%.

## Report summary

**Data and approach**
- Census Monthly Retail Trade Survey, Jan 1992 to Dec 2020, 65 categories, millions of USD, not seasonally adjusted.
- Five categories chosen to contrast trajectories: e-commerce, warehouse clubs (growth), department stores, book stores (decline), gasoline stations (price-driven). All five have complete data (348 months, no nulls).
- Loaded into PostgreSQL (`retail_sales`). All analysis in SQL window functions and regression aggregates; Python for charts and three statistical checks (PELT, Holt-Winters, STL).

**Results**
1. **Trends** (CAGR 1992-2019): e-commerce +11.7%/yr and warehouse clubs +9.8%/yr vs total retail +4.2%; department stores -1.0%/yr and -41.9% from their 2000 peak by 2019; book stores flat overall but -62.6% from their 2007 peak.
2. **Seasonality**: December drives department stores (14.4% of the year); book stores peak in January, August and December; gasoline peaks in summer. The December spike is shrinking in every holiday category.
3. **Change points**: department stores broke in Nov-Dec 2001; book stores in late 2008; gasoline's 9 confirmed turns match oil prices; growth categories show slowdowns (warehouse clubs 13.4% to 9.4% to 3.3% a year).
4. **Forecast**: department stores 2023 = $113B (linear, 95% interval $104B to $121B); methods cluster at $113B to $120B; $96B if 2020 does not recover.
5. **Extended**: e-commerce passed department stores in 2007 and reached 15.2% of retail; recessions permanently lower the declining categories; COVID winners were home-centered and online; one weekend day moves e-commerce YoY by 2.7 points.

**Limitations**
- Nominal dollars (no inflation adjustment); gasoline sales mostly reflect price.
- Not seasonally or trading-day adjusted; single-month YoY values carry calendar noise.
- "E-commerce" is the Census category *Electronic shopping and mail-order houses* (includes catalog sellers; excludes online sales by store-based retailers).
- Forecasts use 10 annual points and assume the 2010-2019 pattern; the series ends in Dec 2020 during COVID.

## Business insights and hypotheses

| # | Hypothesis | Evidence here | How to test |
|---|---|---|---|
| H1 | Department stores lost share to more than e-commerce: warehouse clubs and discount formats took it too | Warehouse clubs gained 7.6 points of share 1992-2009 while department stores lost 4.6; discount dept. stores held (-1.6%) in 2020 while traditional fell -48.6% | Decompose share changes by category; compare discount vs traditional department store series over time |
| H2 | Book stores are a textbook business as much as a gift business | January and August peaks (12.4%, 12.6%) match semester starts | Compare with college enrollment calendars; test whether the Aug/Jan peaks shrank as digital textbooks and rentals grew |
| H3 | The book store collapse was driven by Borders' exit plus e-readers, not the recession alone | Trend broke in late 2008; largest non-COVID shock Dec 2011 (first holiday after Borders closed) | Add e-reader and Amazon book sales data; test the 2011 break with an intervention model |
| H4 | Gasoline station sales measure oil prices more than demand | 9 confirmed turns aligned with oil price moves; lowest correlation with other categories; volatility peaked 2008-2012 | Regress sales on EIA retail gasoline prices; deflate by price to get volume |
| H5 | Warehouse clubs saturated after 2009 | Growth fell to 3.3%/yr; share flat at about 9% since 2009 | Compare store counts and membership data (Costco, Sam's Club, BJ's) |
| H6 | Recessions accelerate structural decline rather than cause temporary dips | Department and book stores never regained pre-2001/2008 levels; growers had no drawdown | Compare pre- and post-recession slopes for more declining categories |
| H7 | Retailers comparing monthly results need calendar adjustment | Weekend composition explains up to a 9-point e-commerce YoY swing; Thanksgiving timing moves Cyber Monday across months | Rebuild YoY on Census seasonally and trading-day adjusted series and compare |
| H8 | Holiday concentration is falling because spending is spread across the year (gift cards, promotions, online) | December share and STL December effect fall in every holiday category; November share did not rise | Test gift card redemption timing (January lift) and promotional calendars |

## Appendix: SQL files
| File | Contents |
|---|---|
| `sql/01_load.sql` | table DDL, load instructions, data checks |
| `sql/02_trends.sql` | `v_monthly_metrics` view: running totals, moving averages, YoY, seasonality |
| `sql/03_change_points.sql` | `v_change_signals`, `v_change_points` views; confirmation logic |
| `sql/04_forecast.sql` | `v_yearly` view; five SQL forecast methods, backtest, sensitivity |
| `sql/05_extended.sql` | `v_drawdown`, `v_calendar` views; ten extended analyses |